# 🗺️ 10.5 DAG & Job Execution

Welcome back! In the previous lesson (**10.4 Lazy Evaluation**), we learned that Spark doesn't execute code immediately. Instead, it lazily builds a blueprint of your instructions called a **DAG** (Directed Acyclic Graph).

But a blueprint is just a drawing on a piece of paper. How does Spark take that drawing and actually organize hundreds of computers to do the physical work? 

In this lesson, we will learn how the **DAG Scheduler** breaks your blueprint down into a strict hierarchy: **Jobs, Stages, and Tasks**. We will also learn about the most dangerous performance killer in Big Data: The **Wide Transformation** (Network Shuffle).

### 🎯 Learning Objectives
* Understand how the **DAG Scheduler** translates logical plans into physical execution.
* Define the Spark execution hierarchy: **Jobs -> Stages -> Tasks**.
* Distinguish between **Narrow Transformations** and **Wide Transformations**.
* Understand what a **Network Shuffle** is and why it creates Stage boundaries.
* See how Data Engineers optimize these execution plans to save computing costs.

## 1. The Execution Hierarchy (From Blueprint to Reality)

When you call an **Action** (like `.show()` or `.write()`), the Driver hands your DAG blueprint to the **DAG Scheduler**. The scheduler acts like a construction foreman, breaking the massive project down into manageable pieces.

Here is the strict hierarchy of how Spark organizes work:

### 🏢 1. Application
Your entire Spark script (from the moment you create the `SparkSession` until you `stop()` it). An application can contain many Jobs.

### 🏗️ 2. Job
A **Job** is triggered every single time you call an **Action**. If your script has three `.show()` commands and one `.write()` command, your Application will create 4 Jobs.

### 🧱 3. Stage
A Job is broken down into **Stages**. A Stage is a sequence of transformations that can be executed perfectly in parallel *without the computers needing to talk to each other*. The moment the computers need to exchange data over the network, a new Stage must be created.

### 👷 4. Task
A Stage is broken down into **Tasks**. A Task is the smallest, atomic unit of work. **One Task processes One Partition of data on One Executor Core.** If you have 100 partitions of data, a stage will launch 100 Tasks.

<img src="../assets/Module_10/10_05_01.png" alt="A hierarchical diagram showing 1 Application breaking down into 2 Jobs. Each Job breaks down into Stages. Each Stage breaks down into many individual Tasks." width="75%">

## 2. Narrow vs. Wide Transformations

To understand why a Job breaks into multiple Stages, we must understand the difference between Narrow and Wide Transformations. This is one of the most frequent interview questions for Data Engineers.

### ➡️ Narrow Transformations (No Shuffle)
In a Narrow Transformation, each partition of data can be processed entirely independently. The Executor does not need to look at any other data on any other server to do its job.
* **Examples:** `.filter()`, `.select()`, `.withColumn()`
* **Analogy:** I give you a bucket of red and blue balls and say, "Filter out the red ones." You can do this by yourself. You don't need to talk to anyone else in the room.
* **Performance:** Extremely Fast. All operations happen in-memory on the local node.

### 🔀 Wide Transformations (The Network Shuffle)
In a Wide Transformation, the data must be reorganized and grouped across the entire cluster. Executors must send data over the network to other Executors.
* **Examples:** `.groupBy()`, `.join()`, `.orderBy()`
* **Analogy:** I give a bucket of unsorted mail to 5 different people in a room. I say, "Group all the mail by Zip Code." To do this, everyone must throw mail across the room to each other to get all the matching zip codes into the same pile.
* **Performance:** Slow and Expensive. Moving data over a network is called a **Shuffle**, and it is the biggest bottleneck in Spark.

<img src="../assets/Module_10/10_05_02.png" alt="Diagram comparing Narrow vs Wide. Left side (Narrow): straight arrows pointing from input partitions to output partitions. Right side (Wide): Criss-crossing arrows showing data moving across the network between different partitions." width="75%">

## 3. How the DAG Scheduler creates Stages

This brings us to the golden rule of Spark Execution:

> 🌟 **A Shuffle always creates a Stage Boundary.** 🌟

The DAG Scheduler will group as many Narrow Transformations together as possible into a single Stage (this is called *Pipelining*). It only stops and creates a new Stage when it hits a Wide Transformation that requires a network Shuffle.

Let's write some code to see this in action!

In [6]:
# 1. Start our Spark Session
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("DAG_Execution").master("local[*]").getOrCreate()

# 2. Create a dummy dataset of e-commerce purchases
data = [
    ("Alice", "Laptop", "USA", 1200),
    ("Bob", "Mouse", "Canada", 25),
    ("Alice", "Keyboard", "USA", 100),
    ("Charlie", "Monitor", "UK", 300),
    ("Bob", "Laptop", "Canada", 1100)
]
columns = ["Name", "Item", "Country", "Price"]
df = spark.createDataFrame(data, columns)

print("Raw Data:")
df.show()

Raw Data:


+-------+--------+-------+-----+
|   Name|    Item|Country|Price|
+-------+--------+-------+-----+
|  Alice|  Laptop|    USA| 1200|
|    Bob|   Mouse| Canada|   25|
|  Alice|Keyboard|    USA|  100|
|Charlie| Monitor|     UK|  300|
|    Bob|  Laptop| Canada| 1100|
+-------+--------+-------+-----+



In [7]:
# --- SCENARIO A: ONLY NARROW TRANSFORMATIONS ---

# .filter() and .select() are NARROW. 
# Each executor can filter its own data without talking to the network.

narrow_df = df.filter(F.col("Price") > 100).select("Name", "Country")

print("--- NARROW TRANSFORMATION EXECUTION PLAN ---")
narrow_df.explain()

# Notice in the output: it says "Filter" and "Project" (Select).
# There is NO "Exchange" mentioned. This entire job runs in ONE single Stage!

--- NARROW TRANSFORMATION EXECUTION PLAN ---
== Physical Plan ==
*(1) Project [Name#25, Country#27]
+- *(1) Filter (isnotnull(Price#28L) AND (Price#28L > 100))
   +- *(1) Scan ExistingRDD[Name#25,Item#26,Country#27,Price#28L]




In [8]:
# --- SCENARIO B: WIDE TRANSFORMATIONS ---

# .groupBy() is WIDE. 
# Spark must shuffle the data so all of "Alice's" records end up on the same machine to be summed up.

wide_df = df.groupBy("Country").sum("Price")

print("\n--- WIDE TRANSFORMATION EXECUTION PLAN ---")
wide_df.explain()

# Look at the output! You will see the word "Exchange hashpartitioning".
# "Exchange" is Spark's word for SHUFFLE. 
# This shuffle forces the DAG Scheduler to split this Job into TWO Stages.


--- WIDE TRANSFORMATION EXECUTION PLAN ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Country#27], functions=[sum(Price#28L)])
   +- Exchange hashpartitioning(Country#27, 200), ENSURE_REQUIREMENTS, [plan_id=84]
      +- HashAggregate(keys=[Country#27], functions=[partial_sum(Price#28L)])
         +- Project [Country#27, Price#28L]
            +- Scan ExistingRDD[Name#25,Item#26,Country#27,Price#28L]




## 4. Tasks (The Atomic Unit of Work)

Inside each Stage, Spark creates **Tasks**. 

How does it know how many tasks to create? 
**Number of Tasks = Number of Data Partitions.**

If you have a 10 GB file split into 100 physical chunks (partitions) across your cluster, Spark will launch exactly 100 Tasks to process that Stage. 

If you only have 2 CPU cores on your laptop, Spark will execute those 100 tasks two at a time until they are all finished.

In [9]:
# Let's check how many Tasks Spark will use to process our DataFrame!
# We do this by checking the number of physical partitions in the underlying RDD.

num_partitions = df.rdd.getNumPartitions()

print(f"Our tiny DataFrame has {num_partitions} partitions.")
print(f"Therefore, any Stage processing this initial data will launch exactly {num_partitions} Tasks!")

# Note: By default, local Spark often sets partitions to the number of CPU cores on your laptop.

Our tiny DataFrame has 12 partitions.
Therefore, any Stage processing this initial data will launch exactly 12 Tasks!


## 🧑‍💻 The Modern Data Ecosystem: Role Comparison

How do different data roles interact with Jobs, Stages, and Tasks?

| Feature | 🏛️ Traditional DBA | 🛠️ Data Engineer (You) | 🧠 Data Scientist |
| :--- | :--- | :--- | :--- |
| **Performance Focus** | Optimizing B-Tree Indexes and SQL Joins on a single server. | **Minimizing Shuffles (Wide Transformations) to keep Jobs down to as few Stages as possible.** | Ensuring the data features are correct for the predictive algorithm. |
| **Troubleshooting** | Checking server CPU/RAM usage logs. | **Reading the Spark UI to find "Straggler Tasks" (one task taking 10x longer than the others due to data skew).** | Checking for overfitting in the ML model. |
| **Cost Optimization** | Buying a slightly larger server. | **Tuning the number of partitions to exactly match the number of Executor cores so no CPU sits idle.** | Not typically responsible for infrastructure costs. |

---

### 📈 Career Progression Roadmap

1. **Junior DE:** Writes PySpark code that successfully transforms data, but uses heavy `.groupBy()` and `.join()` operations carelessly, causing massive network shuffles and high cloud costs.
2. **Mid-Level DE:** Understands **Narrow vs. Wide transformations**. Restructures their code to filter data *before* joining it, significantly reducing the amount of data moving across the network during a Shuffle.
3. **Senior DE:** Masters the **Spark UI**. A Senior DE can look at a graphical DAG, identify a Stage that is causing a bottleneck, and completely re-architect the data pipeline to bypass the Shuffle entirely (e.g., using a Broadcast Join, which we will learn later).

### 🛠️ Skills Matrix
* **Core Concept:** Execution Hierarchy (Job -> Stage -> Task).
* **Performance Concept:** Network Shuffles (caused by Wide Transformations).
* **Diagnostic Tool:** Spark UI & `.explain()` physical plans.

## 🔑 Key Takeaways

- **Hierarchy:** Application -> Jobs (Triggered by Actions) -> Stages (Separated by Shuffles) -> Tasks (Mapped to Partitions).
- **Narrow Transformations:** e.g., `.filter()`. Fast. Data stays on the local Executor. No network traffic.
- **Wide Transformations:** e.g., `.groupBy()`. Slow. Data must be reorganized across the cluster. This creates a **Shuffle**.
- **Stage Boundaries:** Every time Spark hits a Shuffle, it is forced to create a new Stage.
- **Tasks:** The ultimate unit of work. One task processes one partition of data on one CPU core.

## 📝 Knowledge Check Quiz

**Question 1:** In the Spark execution hierarchy, what exactly triggers the creation of a new **Job**?
A) A Narrow Transformation.
B) Creating a SparkSession.
C) Calling an Action (like `.show()`, `.count()`, or writing data to disk).
D) A Network Shuffle.

**Question 2:** You write a PySpark script and the `.explain()` plan shows an `Exchange` step. What does this mean for your cluster?
A) It means Spark has encountered a Wide Transformation (like a groupBy) and must Shuffle data across the network, which is expensive and creates a new Stage.
B) It means Spark is exchanging data with Hadoop MapReduce.
C) It means the code is perfectly optimized and using a Narrow Transformation.
D) It means the Driver node has crashed and is exchanging roles with a Worker.

**Question 3:** If you have a DataFrame that is physically split into 500 partitions, how many **Tasks** will Spark generate for a Stage processing that DataFrame?
A) 1 Task
B) Depends on the number of CPU cores.
C) Exactly 500 Tasks (1 Task per partition).
D) 50 Tasks

---

*Answers: 1: C, 2: A, 3: C*

In [10]:
# Clean up our session
spark.stop()
print("Spark session closed.")

Spark session closed.


### 🚀 What's Next?
We just discovered that the number of **Tasks** Spark runs is entirely dependent on the number of **Partitions** your data is split into. 

If you don't control your partitions, you don't control your cluster's performance! In the next lesson, **10.6 Spark Partitions Fundamentals**, we will dive deep into how Spark manages these physical chunks of data and how you can optimize them.